# PSPI: Prokaryote SP Identifier

A model using long short-term memory (LSTM) and (n,k)-mers to identify prokaryotic signal peptides (SPs).

This notebook processes protein sequences to predict whether they contain signal peptides using a trained LSTM model with feature vectors and gap-dimer encoding.

## Import Required Libraries

In [1]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Activation
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import pickle
import numpy as np
import os
import sys
import random
import csv
from pathlib import Path

# Configuration
timesteps = 1
print("Libraries imported successfully!")

2026-03-28 10:18:26.811838: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Libraries imported successfully!


In [2]:
# Create filename based on current timestamp
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_FILE = f"results_{timestamp}.csv"
print(f"Results will be saved to: output_files/{RESULTS_FILE}")

Results will be saved to: output_files/results_20260328_101831.csv


## Feature Extraction Functions

These functions extract amino acid vectors and gap-dimer features from protein sequences.

In [3]:
def get_aavectors(seq_temp):
    """Convert amino acid sequence to one-hot encoded vector."""
    seq = seq_temp
    fea = []
    tem_vec = []
    amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
    aa_dict = {aa: i for i, aa in enumerate(amino_acids)}
    
    k = len(seq_temp)
    for i in range(k):
        if seq[i] in aa_dict:
            tem_vec = [0] * 20
            tem_vec[aa_dict[seq[i]]] = 1
        else:
            tem_vec = [0] * 20
        fea = fea + tem_vec
    
    # Pad to 100 amino acids
    for i in range(k, 100):
        fea = fea + [0] * 20
    
    return fea

def get_gap_dimer(seq):
    """Extract gap-dimer features from sequence."""
    chars = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
    chardict = {chars[i]: i for i in range(20)}
    count = [0] * 400
    motif_size = 4
    
    for i in range(0, len(seq) - 1):
        for j in range(i + 1, min(i + motif_size, len(seq))):
            try:
                index = chardict[seq[i]] * 20 + chardict[seq[j]]
                count[index] += 1
            except:
                continue
    
    return count

## Sequence Processing Functions

In [4]:
def prepare_feature(sequences):
    """Prepare feature vectors from protein sequences."""
    protein_seq_dict = {}
    protein_index = 1
    
    for line in sequences:
        seq = line
        protein_seq_dict[protein_index] = seq
        protein_index = protein_index + 1
    
    aavectors = []
    gapDimer = []
    
    # Get protein features
    for i in protein_seq_dict:
        aavectors_feature = get_aavectors(protein_seq_dict[i])
        gapDimer_feature = get_gap_dimer(protein_seq_dict[i])
        
        if len(aavectors_feature) != 20 * 100:
            print(f"Warning: {protein_seq_dict[i]} is too long and will be discarded")
            continue
        
        aavectors.append(aavectors_feature)
        gapDimer.append(gapDimer_feature)
    
    X = np.concatenate((aavectors, gapDimer), axis=1)
    return X

def getSeqs(dir):
    """Read sequences from FASTA files in directory."""
    seq = []
    try:
        directory = os.listdir(dir)
        for file in directory:
            if file.endswith(('.fasta', '.fas', '.fa')):
                with open(os.path.join(dir, file), 'r') as f:
                    fileList = f.readlines()
                    for line in fileList:
                        if line[0] != '>':
                            seq.append(line.rstrip())
    except FileNotFoundError:
        print(f"Directory not found: {dir}")
    return seq

## Model Evaluation Functions

In [5]:
def calculate_performace(test_num, pred_y, labels):
    """Calculate performance metrics (precision, sensitivity, specificity, F1)."""
    tp = 0
    fp = 0
    tn = 0
    fn = 0
    
    for index in range(test_num):
        if labels[index] == 1:
            if labels[index] == pred_y[index]:
                tp = tp + 1
            else:
                fn = fn + 1
        else:
            if labels[index] == pred_y[index]:
                tn = tn + 1
            else:
                fp = fp + 1
    
    precision = float(tp) / (tp + fp) if (tp + fp) > 0 else 0
    sensitivity = float(tp) / (tp + fn) if (tp + fn) > 0 else 0
    specificity = float(tn) / (tn + fp) if (tn + fp) > 0 else 0
    f1 = float(2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    
    return precision, sensitivity, specificity, f1

def transfer_label_from_prob(proba, threshold=None):
    """Convert probability predictions to binary labels."""
    if threshold is not None:
        label = [1 if val >= threshold else 0 for val in proba]
    else:
        label = [1 if val >= 0.75 else 0 for val in proba]
    return label

def plot_roc_curve(labels, probality):
    """Calculate ROC AUC score."""
    fpr, tpr, thresholds = roc_curve(labels, probality)
    roc_auc = auc(fpr, tpr)
    return roc_auc

## Model Training

Build and train an LSTM model on the training data.

In [6]:
def buildModel():
    """Build and train the LSTM model."""
    batch_size = 32
    epochs = 30
    
    labeltraining = []
    print("Reading Training data")
    print("-" * 50)
    
    # Load positive training sequences
    sequences = getSeqs("datasets/training/positives/")
    for i in range(0, len(sequences)):
        labeltraining.append(1)
    
    # Load negative training sequences
    sequences += getSeqs("datasets/training/negatives/")
    for i in range(len(labeltraining), len(sequences)):
        labeltraining.append(0)
    
    print(f"Loaded {len(sequences)} training sequences")
    print("Preparing features")
    print("-" * 50)
    
    aavectorstraining = prepare_feature(sequences)
    X_train = np.array(np.reshape(aavectorstraining, 
                                   (len(aavectorstraining), timesteps, len(aavectorstraining[0]))))
    train_label = np.array(labeltraining)
    
    # Build model
    model = Sequential()
    model.add(LSTM(128, return_sequences=False, 
                   input_shape=(timesteps, len(aavectorstraining[0])), name='lstm1'))
    model.add(Dropout(0.25, name='dropout'))
    model.add(Dense(1, name='full_connect2'))
    model.add(Activation('sigmoid'))
    
    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    
    print("Training the model")
    print("-" * 50)
    model.fit(X_train, train_label, batch_size=batch_size, epochs=epochs, verbose=1)
    
    return model

## Model Testing

Test the model on test data with multiple random samples.

In [7]:
def testModel(model, threshold=None):
    """Test model and compute metrics over 12 trials."""
    print("Testing")
    print("-" * 50)
    
    pos_sequences = getSeqs("datasets/testing/positives/")
    neg_sequences = getSeqs("datasets/testing/negatives/")
    
    print(f"Available test sequences: {len(pos_sequences)} positive, {len(neg_sequences)} negative")
    
    sums = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
    
    for j in range(1, 13):
        subsequence = []
        labeltesting = []
        y_pred = []
        all_prob = []
        
        # Sample or use all positive sequences
        if len(pos_sequences) > 1000:
            subsequence += random.sample(pos_sequences, 1000)
            labeltesting += [1] * 1000
        else:
            subsequence += pos_sequences
            labeltesting += [1] * len(pos_sequences)
        
        # Sample or use all negative sequences
        if len(neg_sequences) > 1000:
            subsequence += random.sample(neg_sequences, 1000)
            labeltesting += [0] * 1000
        else:
            subsequence += neg_sequences
            labeltesting += [0] * len(neg_sequences)
        
        # Prepare features and make predictions
        aavectorstesting = prepare_feature(subsequence)
        X_test_all = np.array(np.reshape(aavectorstesting, 
                                          (len(aavectorstesting), timesteps, len(aavectorstesting[0]))))
        lstm_proba = model.predict(X_test_all, verbose=0)
        y_pred = transfer_label_from_prob(lstm_proba, threshold)
        all_prob = [val for val in lstm_proba]
        
        # Calculate metrics
        test_label = np.array(labeltesting)
        prec, sensitivity, specificity, f1 = calculate_performace(len(test_label), y_pred, test_label)
        roc_auc = plot_roc_curve(test_label, all_prob)
        
        precision, recall, t = precision_recall_curve(test_label, all_prob)
        aupr = auc(recall, precision)
        
        print(f"Results: Trial {j}/12")
        print(f"precision\t{round(prec, 3)}")
        print(f"sensitivity\t{round(sensitivity, 3)}")
        print(f"specificity\t{round(specificity, 3)}")
        print(f"f1\t\t{round(f1, 3)}")
        print(f"AUROC\t\t{round(roc_auc, 3)}")
        print(f"AUPR\t\t{round(aupr, 3)}")
        
        sums += np.array([prec, sensitivity, specificity, f1, roc_auc, aupr])
        print("-" * 50)
    
    print("Results: Average")
    print(f"precision\t{round(sums[0]/12, 3)}")
    print(f"sensitivity\t{round(sums[1]/12, 3)}")
    print(f"specificity\t{round(sums[2]/12, 3)}")
    print(f"f1\t\t{round(sums[3]/12, 3)}")
    print(f"AUROC\t\t{round(sums[4]/12, 3)}")
    print(f"AUPR\t\t{round(sums[5]/12, 3)}")
    print("-" * 50)

## Prediction Functions

Functions to run predictions on input files and save results.

In [8]:
def readFile(model, inFile, outfile, threshold):
    """Process input file and write predictions."""
    labels = []
    sequences = []
    
    for line in inFile:
        if line[0] == ">":
            labels.append(line.rstrip())
        else:
            sequences.append(line.rstrip())
        
        if len(sequences) == 1000:
            features = prepare_feature(sequences)
            X_test_all = np.array(np.reshape(features, 
                                               (len(features), timesteps, len(features[0]))))
            lstm_proba = model.predict(X_test_all, verbose=0)
            y_pred = transfer_label_from_prob(lstm_proba, threshold)
            
            for i in range(0, len(sequences)):
                outfile.writerow([labels[i], sequences[i], y_pred[i], lstm_proba[i][0]])
            
            labels = []
            sequences = []
    
    # Process remaining sequences
    if sequences:
        features = prepare_feature(sequences)
        X_test_all = np.array(np.reshape(features, 
                                           (len(features), timesteps, len(features[0]))))
        lstm_proba = model.predict(X_test_all, verbose=0)
        y_pred = transfer_label_from_prob(lstm_proba, threshold)
        
        for i in range(0, len(sequences)):
            outfile.writerow([labels[i], sequences[i], y_pred[i], lstm_proba[i][0]])

def categorize(model, input_path, output, threshold):
    """Run prediction on input files."""
    # Create output directory if it doesn't exist
    Path("output_files").mkdir(exist_ok=True)
    
    if output is not None:
        outfile = csv.writer(open("output_files/" + output, 'w'))
    else:
        outfile = csv.writer(open(f"output_files/{RESULTS_FILE}", 'w'))
    
    outfile.writerow(["Seq name", "sequence", "prediction", "confidence"])
    
    if input_path is not None:
        print(f"Processing file: {input_path}")
        with open(input_path, "r") as inFile:
            readFile(model, inFile, outfile, threshold)
    else:
        files = os.listdir("input_files")
        for file in files:
            print(f"Processing file: {file}")
            with open("input_files/" + file, "r") as inFile:
                readFile(model, inFile, outfile, threshold)

## Main PSPI Function

Orchestrates the complete workflow: training, testing, or inference.

In [9]:
def PSPI(model_path=None, train_model=None, test_model=None, 
          file_input=None, output=None, threshold=None):
    """
    Main PSPI function.
    
    Parameters:
    -----------
    model_path : str, optional
        Path to a pre-trained model
    train_model : str, optional
        Name for new model to train
    test_model : str, optional
        Path to model to test
    file_input : str, optional
        Path to specific input file
    output : str, optional
        Output filename
    threshold : float, optional
        Classification threshold (0-1)
    """
    
    if train_model is not None:
        print("\n=== TRAINING NEW MODEL ===")
        model = buildModel()
        testModel(model, threshold)
        
        yes_res = ['y', 'yes']
        no_res = ['n', 'no']
        
        while True:
            user_input = input('Save the model? y/n: ')
            if user_input.lower() in yes_res:
                with open(train_model, "wb") as f:
                    pickle.dump(model, f)
                print(f"Model saved to {train_model}")
                return
            elif user_input.lower() in no_res:
                print("Model not saved.")
                return
            else:
                print('Response must be yes or no')
    
    if test_model is not None:
        print("\n=== TESTING EXISTING MODEL ===")
        model = pickle.load(open(test_model, "rb"))
        testModel(model, threshold)
        return
    
    # Load model for inference
    if model_path is not None:
        print(f"\n=== LOADING MODEL: {model_path} ===")
        model = pickle.load(open(model_path, "rb"))
    else:
        print("\n=== LOADING DEFAULT MODEL ===")
        model = pickle.load(open("model_default.pkl", "rb"))
    
    print("=== RUNNING PREDICTION ===")
    categorize(model, file_input, output, threshold)

## Usage Examples

### Example 1: Run inference on all files in input_files/ directory
```python
PSPI()
```

### Example 2: Run inference with custom model
```python
PSPI(model_path="my_model.pkl", threshold=0.5)
```

### Example 3: Train a new model
```python
PSPI(train_model="my_new_model.pkl")
```

### Example 4: Test an existing model
```python
PSPI(test_model="my_model.pkl")
```

### Example 5: Process a specific input file
```python
PSPI(model_path="my_model.pkl", file_input="test.fasta", output="predictions.csv")
```

## Quick Start

Uncomment and run one of the examples below to perform the desired operation.

In [10]:
# Example: Run inference on all input files with default model
# PSPI()

# Example: Train a new model
# PSPI(train_model="my_model.pkl")

PSPI(train_model="berns.pkl")

# Example: Test a trained model
# PSPI(test_model="my_model.pkl")

# Example: Run inference with custom threshold
# PSPI(threshold=0.5)

print("Ready to run PSPI! Uncomment a function call in the cell above to begin.")


=== TRAINING NEW MODEL ===
Reading Training data
--------------------------------------------------
Loaded 29925 training sequences
Preparing features
--------------------------------------------------
Training the model
--------------------------------------------------
Epoch 1/30
936/936 [==============================] - 15s 14ms/step - loss: 0.1218 - accuracy: 0.9578
Epoch 2/30
936/936 [==============================] - 15s 16ms/step - loss: 0.0652 - accuracy: 0.9773
Epoch 3/30
936/936 [==============================] - 19s 20ms/step - loss: 0.0319 - accuracy: 0.9902
Epoch 4/30
936/936 [==============================] - 23s 25ms/step - loss: 0.0131 - accuracy: 0.9966
Epoch 5/30
936/936 [==============================] - 20s 21ms/step - loss: 0.0052 - accuracy: 0.9991
Epoch 6/30
936/936 [==============================] - 20s 21ms/step - loss: 0.0018 - accuracy: 0.9999
Epoch 7/30
936/936 [==============================] - 23s 25ms/step - loss: 8.1707e-04 - accuracy: 1.0000
Epoch 8/3